In [ ]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Add it to your .env file.")

genai.configure(api_key=gemini_api_key)
gemini_client = genai.GenerativeModel("gemini-2.5-flash")

print("API client successfully configured")


In [ ]:
from IPython.display import display, Markdown
import json
import re

def print_markdown(text):
    display(Markdown(text))

def parse_json_response(raw_json_text):
    cleaned = raw_json_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.IGNORECASE | re.MULTILINE).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def display_nutrition_card(raw_json_text):
    try:
        data = parse_json_response(raw_json_text)
    except json.JSONDecodeError as error:
        print_markdown(
            "### Could not parse the nutrition JSON\n\n"
            f"JSON error: `{error}`\n\n"
            "Raw Gemini response:\n\n"
            f"```text\n{raw_json_text}\n```"
        )
        return
    markdown = f"""
## {data.get('food_name', 'Food analysis')}

**Serving:** {data.get('serving_description', 'Unknown')}  
**Calories:** {data.get('calories', 'Unknown')} kcal  
**Protein:** {data.get('protein_grams', 'Unknown')} g  
**Fat:** {data.get('fat_grams', 'Unknown')} g  
**Confidence:** {data.get('confidence_level', 'Unknown')}

> Estimates from an image are approximate. Weigh the portion for tracking accuracy.
"""
    display(Markdown(markdown))

In [ ]:
from PIL import Image
image_filename="food_image.jpg"
img= Image.open(image_filename)
print(f"Image '{image_filename}' loaded successfully!")
print(f"Format: {img.format}")
print(f"Size: {img.size}")
print(f"Mode: {img.mode}")

display(img)

image_to_analyze= img

In [ ]:
import io
import base64

In [ ]:
def encode_image_to_base64(image_path_or_pil):
    if isinstance(image_path_or_pil, str):
        if not os.path.exists(image_path_or_pil):
            raise FileNotFoundError(f"Image file not found at: {image_path_or_pil}")
        with open(image_path_or_pil, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")
        
    elif isinstance(image_path_or_pil, Image.Image):
        buffer = io.BytesIO()
        image_format = image_path_or_pil.format or "JPEG"
        image_path_or_pil.save(buffer, format=image_format)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")
    
    else:
        raise ValueError("Input must be a file path (str) or a PIL Image object")
        

In [ ]:
def query_geminiai_vision(client, image, prompt, max_tokens=800, temperature=0.35):
    try:
        response = client.generate_content(
            [prompt, image],
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=max_tokens,
                temperature=temperature,
            ),
        )
        return response.text
    except Exception as e:
        return f"Error calling API: {e}"

In [ ]:
food_recognition_prompt= """
Analyze this food image for a friendly calorie-tracking notebook.

Return Markdown, not JSON. Make it easy to scan:
- Start with a short title using the most likely food name.
- Add a one-sentence visual description.
- Add 3 bullets: likely ingredients, calorie estimate, and macro estimate.
- Add one short note about uncertainty or portion size.

Keep it useful, warm, and concise. Avoid sounding like a database row.
"""
print(f"{food_recognition_prompt}")

In [ ]:
print("Querying Gemini Vision...")
gemini_description= query_geminiai_vision(
    gemini_client,
    image_to_analyze,
    food_recognition_prompt
)
print_markdown(gemini_description)

In [ ]:
structured_nutrition_prompt = """
# Nutritional Analysis Task

## Context
You are a nutrition expert analyzing food images to provide accurate nutritional information.

## Instructions
Analyze the food item in the image and provide estimated nutritional information based on your knowledge.

## Input
- An image of a food item

## Output
Provide the following estimated nutritional information for a typical serving size or per 100g:
- food_name (string)
- serving_description (string, e.g., '1 slice', '100g', '1 cup')
- calories (float)
- fat_grams (float)
- protein_grams (float)
- confidence_level (string: 'High', 'Medium', or 'Low')

**IMPORTANT:** Respond ONLY with a single JSON object containing these fields. Do not include any other text, explanations, or apologies. The JSON keys must match exactly: "food_name", "serving_description", "calories", "fat_grams", "protein_grams", "confidence_level". If you cannot estimate a value, use `null`.

Example valid JSON response:
{
  "food_name": "Banana",
  "serving_description": "1 medium banana (approx 118g)",
  "calories": 105.0,
  "fat_grams": 0.4,
  "protein_grams": 1.3,
  "confidence_level": "High"
}
"""


In [ ]:
# Call Gemini with the structured prompt, then render the JSON as a readable card.
gemini_nutrition_result = query_geminiai_vision(client = gemini_client,
                                              image = image_to_analyze,
                                              prompt = structured_nutrition_prompt,
                                              max_tokens = 1000,
                                              temperature = 0.1)

display_nutrition_card(gemini_nutrition_result)